In [ ]:
import torch 
import numpy as np 
import matplotlib.pyplot as plt
from itertools import islice
import json
from tqdm.auto import tqdm

from cryo_sbi.wpa_simulator.cryo_em_simulator import cryo_em_simulator
from cryo_sbi import CryoEmSimulator
from cryo_sbi.inference.models import build_models
import cryo_sbi.utils.estimator_utils as est_utils
from cryo_sbi.inference.priors import get_image_priors, PriorLoader
from cryo_sbi.inference.models.build_models import build_nle_flow_model

### Testing the NLE flow

Here we test the quality of the NLE flow. First we need to load the trained model.

In [ ]:
# Parameters
device='cuda:0'
# Load trained model
estimator = est_utils.load_estimator(
    "training_parameters_nle.json",
    build_models.build_nle_flow_model,
    "tutorial_estimator.pt",
    device=device,
)

### Image generation for testing

Now we reload the orginal models in `hsp90_models.pt`, center them, and create a new set of models containing the close and open states in the desired population. We start with a 50%-50% mixture of close and open states 

In [ ]:
def center_models(models):
    """
    Remove center of mass from each model.
    
    Args:
        models: torch.Tensor of shape [num_models, 3, N]
                where 3 = (x, y, z) and N = number of atoms
    
    Returns:
        centered_models: torch.Tensor of same shape, centered at origin
    """
    # Compute center of mass for each model
    # Mean over atoms (dim=2) -> [num_models, 3]
    com = models.mean(dim=2, keepdim=True)  # [num_models, 3, 1]
    
    # Subtract center of mass
    centered_models = models - com
    
    return centered_models

In [ ]:
# load models from file
models = torch.load("../hsp90_models.pt").to(device)
# center models (just to be sure)
models = center_models(models)
# select close and open states
first = models[0].unsqueeze(0).repeat(1, 1, 1)
last  = models[-1].unsqueeze(0).repeat(1, 1, 1)
# create desired mixture
models = torch.cat([first, last], dim=0)
# save - needed for image generation
torch.save(models, "models.pt")

We will now simulate the cryo-EM images with our generated models. The simulation is done by the class `CryoEmSimulator` and the simulation is run by the `simulate` function. The class `CryoEmSimulator` takes as input a config file with the simulation parameters. The config file used here is `simulation_parameters.json`.

The following parameters are used in the simulation:
```
{
    "N_PIXELS": 128,
    "PIXEL_SIZE": 1.5,
    "SIGMA": [0.5, 5.0],
    "MODEL_FILE": "models.pt",
    "SHIFT": 0.0,
    "DEFOCUS": [0.5, 2.0],
    "SNR": [0.01, 0.5],
    "AMP": 0.1,
    "B_FACTOR": [1.0, 100.0]
}
```

In [ ]:
# create simulator
simulator = CryoEmSimulator(
    "simulation_parameters.json"
)  # creating simulator with simulation parameters
# generate 100k images
images, parameters = simulator.simulate(
    num_sim=100000, return_parameters=True
)  # simulating images and save parameters
# be sure images are on device
images = images.to(device)

#### Visualize the simulated images

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(10, 10), sharex=True, sharey=True)
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(images[idx], vmin=-3, vmax=3, cmap="gray")
    ax.axis("off")

### Evaluate likelihood matrix

In [ ]:
def evaluate_likelihood_pairwise(
    estimator: torch.nn.Module,
    images: torch.Tensor,        # shape: [N_images, H, W]
    models: torch.Tensor,        # shape: [N_models, 3, N_atoms] - same as training
    batch_size_images: int = 64,
    device: str = "cpu"
) -> torch.Tensor:
    """
    Evaluate log p(X_i | theta_j) for all pairs (i, j).
    Models are the same as used in training, so indices = 0, 1, 2, ..., N_models-1
    
    Args:
        estimator: trained NLE model
        images: tensor of images
        models: tensor of models (same as training, same order)
        batch_size_images: batch size for images
        batch_size_models: batch size for models
        device: computation device
    
    Returns:
        log_probs: shape [N_images, N_models] 
                   log_probs[i, j] = log p(image_i | model_j)
    """
    estimator.eval()
    estimator.to(device)
    
    N_images = len(images)
    N_models = len(models)
    
    # Model indices are simply 0, 1, 2, ..., N_models-1
    model_indices = torch.arange(N_models, dtype=torch.long).to(device)
    
    # Initialize results matrix
    log_probs = torch.zeros(N_images, N_models).to(device)
    
    with torch.no_grad():
        # Iterate over image batches
        for i in tqdm(range(0, N_images, batch_size_images), desc="Images"):
            i_end = min(i + batch_size_images, N_images)
            batch_images = images[i:i_end]  # [B_img, H, W]
            
            # Iterate over all models
            for j in range(0, N_models):            
                # Evaluate likelihood
                log_probs[i:i_end, j] = estimator(batch_images, model_indices[j:j+1])
    
    return log_probs

Before evaluating the likelihood, we reload all the 20 models. No need to center them, only the indexes matter, as the flow has learned the relationship between a cryo-EM image and the model index (from 0 to 19).

In [ ]:
# load models from file
models = torch.load("../hsp90_models.pt").to(device)

In [ ]:
# evaluate likelihood matrix
log_probs_matrix = evaluate_likelihood_pairwise(
    estimator,
    images,
    models,
    batch_size_images=100,
    device=device
)
# transpose to [N_images : N_models]
log_probs_matrix = log_probs_matrix.T

### Population inference (BioEM style)

Now we can finally infer the population of the 20 original models using the usual BioEM/CryoLike formula. We expect only the first and last to be populated at 50%-50% as per our design!

In [ ]:
# minimization functions - in italian ;-)
def objective(y, r):
    """
    y: torch.Tensor shape (n,) (parametri liberi)
    r: torch.Tensor shape (n, m) (log-likelihood)
    """
    x = torch.softmax(y, dim=0)  # vettore nel simplex
    log_x = torch.log(x + 1e-16)  # stabilità
    log_terms = torch.logsumexp(log_x[:, None] + r, dim=0)  # log(sum_i x_i * exp(r_ij))
    return -torch.sum(log_terms)

def ottimizza(r, sensibilita=1e-10, pazienza=100, max_iter=50000, verbose=True):
    """
    Esegue l'ottimizzazione con early stopping.
    
    Parametri:
        r: tensore di input
        sensibilita: miglioramento minimo richiesto nella loss per continuare
        pazienza: numero di iterazioni consecutive senza miglioramento dopo cui fermarsi
        max_iter: numero massimo di iterazioni
    """
    r = r.to(device=device)
    n = r.shape[0]
    y = torch.zeros(n, device=device, requires_grad=True)
    
    optimizer = torch.optim.Adam([y], lr=0.05)
    history = []
    
    best_loss = float('inf')
    counter = 0  # conta quante iterazioni senza miglioramento
    
    for it in range(max_iter):
        optimizer.zero_grad()
        loss = objective(y, r)
        loss.backward()
        optimizer.step()
        
        current_loss = loss.item()
        history.append(current_loss)
        
        # Controllo miglioramento
        if best_loss - current_loss > sensibilita:
            best_loss = current_loss
            counter = 0  # reset contatore
        else:
            counter += 1
        
        # Stampa di debug
        if ((it % 50 == 0) and (verbose)):
            print(f"Iter {it:4d} | Loss {current_loss:.6f} | Best {best_loss:.6f} | No improv: {counter}")
        
        # Early stopping
        if counter >= pazienza:
            if verbose:
                print(f"Early stopping a iterazione {it} — nessun miglioramento in {pazienza} step.")
            break
    
    return y, history

In [ ]:
# do optimization
ys, _ = ottimizza(log_probs_matrix,verbose=True)
# optimal weights
x_opt = torch.softmax(ys, dim=0).detach().cpu().numpy()

And now we can plot the weights

In [ ]:
# Calculate populations
n_mod = len(x_opt)
mid_point = int(n_mod / 2)
p_A = np.sum(x_opt[0:mid_point])
p_B = np.sum(x_opt[mid_point:])

# Create bar plot with indexes from 1 to 20
fig, ax = plt.subplots(figsize=(10, 5))
indexes = np.arange(1, n_mod + 1)  # 1 to 20
ax.bar(indexes, x_opt, width=0.8, alpha=0.7, edgecolor='black')

# Add vertical dashed line at 10.5 (between bars 10 and 11)
ax.axvline(x=10.5, color='red', linestyle='--', linewidth=2, label='Boundary')

# Add text labels for populations
# Left region (p_A)
ax.text(5.5, ax.get_ylim()[1] * 0.9, f'p_A = {p_A:.3f}', 
        fontsize=14, ha='center', weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Right region (p_B)
ax.text(15.5, ax.get_ylim()[1] * 0.9, f'p_B = {p_B:.3f}', 
        fontsize=14, ha='center', weight='bold',
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

# Labels and styling
ax.set_xlabel('Model Index', fontsize=12)
ax.set_ylabel('Weight', fontsize=12)
ax.set_title('Final Weights Distribution', fontsize=14, weight='bold')
ax.set_xlim(0, 21)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Population A: {p_A:.4f}, Population B: {p_B:.4f}")